# HDFC Bank: Fraud Risk Analysis - Part 2

**Objective:** Feature Engineering and Preprocessing Pipeline

In this notebook, we move from data understanding to data preparation. We will use the robust `scikit-learn` pipeline we built in our `fraudguard` package to transform the raw historical CSV into model-ready numerical matrices.

Crucially, we must avoid **Data Leakage**. We will temporally split our data first, and then `.fit()` our pipeline *only* on the past training data, before we `.transform()` the future validation data.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
from fraudguard.data.ingestion import load_bank_data, split_temporal
from fraudguard.features.engineering import build_feature_pipeline

# 1. Load the Data Warehouse Extract
data_dir = Path.cwd().parent / "data" / "raw"
print("Loading HDFC transaction data warehouse extract...")
df = load_bank_data(data_dir)

Loading HDFC transaction data warehouse extract...


## 1. Temporal Splitting (Preventing Future Leakage)
We use our custom `split_temporal` function which sorts transactions by time. We train on the oldest 80%, and validate on the newest 20%.

In [2]:
# We must split the data before doing any imputation or scaling!
df_train, df_test = split_temporal(df, test_ratio=0.2)

print(f"Training set shape (Past 80%): {df_train.shape}")
print(f"Testing set shape (Future 20%): {df_test.shape}")

# Separate Features (X) and Target (y)
target = 'isFraud'
X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_test = df_test.drop(columns=[target])
y_test = df_test[target]

Training set shape (Past 80%): (472432, 434)
Testing set shape (Future 20%): (118108, 434)


## 2. Defining the Feature Sets
Let's pick a strong subset of features for our baseline models. We will grab the transaction amounts, engineered Vesta features, and a few categorical networks.

In [3]:
# Let's select a representative subset for our initial pipeline
numeric_features = [
    'TransactionAmt', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'D1', 'D2', 'D3',
    'V1', 'V2', 'V3', 'V4', 'V5'
]

categorical_features = [
    'ProductCD', 'card4', 'card6', 'P_emaildomain'
]

print(f"Selected {len(numeric_features)} numeric features and {len(categorical_features)} categorical features.")

Selected 14 numeric features and 4 categorical features.


## 3. Applying the Scikit-Learn Pipeline
We use our custom `build_feature_pipeline()` which chains:
1. Custom Domain Features (e.g., Time of Day)
2. Imputation (Median for numeric, 'Missing' for categorical)
3. Scaling (StandardScaler)
4. Encoding (OneHotEncoder)

In [4]:
# Build the pipeline using our package
pipeline = build_feature_pipeline(numeric_features, categorical_features)

# Fit on training data ONLY to learn medians and standard deviations
print("Fitting pipeline on training data...")
X_train_processed = pipeline.fit_transform(X_train)

# Transform the test data using the statistics learned from the training data
print("Transforming test data...")
X_test_processed = pipeline.transform(X_test)

print(f"\nPipeline execution complete!")
print(f"Original X_train shape: {X_train.shape}")
print(f"Processed X_train matrix shape: {X_train_processed.shape}")

Fitting pipeline on training data...
Transforming test data...

Pipeline execution complete!
Original X_train shape: (472432, 433)
Processed X_train matrix shape: (472432, 89)


## 4. Inspecting the Transformed Output
Let's look at the final numerical tensor that will be fed into our Machine Learning models.

In [5]:
print("Preview of the first 3 rows and first 10 columns of the model-ready matrix:")
print(np.round(X_train_processed[:3, :10], 3))

print("\nNotice how all values are now scaled floats, and categorical variables have been converted into 0.0/1.0 indicators!")

Preview of the first 3 rows and first 10 columns of the model-ready matrix:
[[-0.277 -0.141 -0.091 -0.097 -0.092 -0.036 -0.499 -0.279 -0.131  0.006]
 [-0.444 -0.183 -0.091 -0.097 -0.092 -0.036 -0.591 -0.279 -0.237  0.006]
 [-0.317  0.969 -0.091 -0.097 -0.092 -0.036 -0.591 -0.279 -0.237  0.006]]

Notice how all values are now scaled floats, and categorical variables have been converted into 0.0/1.0 indicators!
